# UVERA · NB99 — Evaluation, Fairness & Packaging

| | |
|---|---|
| **Who runs it** | Rubab (P1) |
| **Accelerator** | None (CPU) |
| **Internet** | On |
| **Secret** | — |
| **Inputs to add** | — (reads every `reports/metrics_ai*.json` committed in the repo; run **last**) |
| **Expected runtime** | ~2–5 min |
| **How to run** | **Save Version → Save & Run All (Commit)** — runs in the background, no 20-minute idle timeout, 12-hour cap |

**What this AI does (say this to judges):** Collects every measured result into one summary for the website's Trust Center, the README and the report: model vs baseline, calibration, 'not sure' rates, fairness slices, ablations, and forecast backtests. Anything not measured is shown as 'not measured yet', never as a made-up number.

**Outputs** (in `/kaggle/working/outputs/NB99_outputs.zip`): `reports/metrics.json`, `reports/summary.md`, `reports/figures/summary_recall_vs_rule.png`

All data is synthetic or open-licensed. No real customer data.

In [ ]:
# 1) Get the UVERA code from the public GitHub repo (nothing to upload by hand)
REPO = "https://github.com/ABRUBAB/takaresolve-ai-hackathon-2026.git"
REF = "main"   # replace with a commit hash to reproduce an exact run
import os, sys, subprocess
if not os.path.exists("/tmp/uvera"):
    subprocess.run(["git", "clone", "-q", REPO, "/tmp/uvera"], check=True)
subprocess.run(["git", "-C", "/tmp/uvera", "fetch", "-q", "origin", REF], check=False)
subprocess.run(["git", "-C", "/tmp/uvera", "checkout", "-q", REF], check=True)
COMMIT = subprocess.run(["git", "-C", "/tmp/uvera", "rev-parse", "HEAD"], capture_output=True, text=True).stdout.strip()
os.environ["UVERA_ROOT"] = "/tmp/uvera"
sys.path.insert(0, "/tmp/uvera/ml")   # use Kaggle's own numpy/pandas/sklearn/lightgbm (no reinstall)
print("UVERA code at commit", COMMIT)

In [ ]:
# 2) Reproducibility: seeds, versions, time budget, output folder
import json
from IPython.display import Image, display
from uvera_ml.common import env_info, set_seed, Budget, out_dir, zip_outputs, write_json
set_seed(42)
budget = Budget(minutes=30)
OUT = out_dir("NB99")
info = env_info(); info["uvera_commit"] = COMMIT
write_json(OUT / "reports" / "versions_nb99.json", info)
print(json.dumps(info, indent=1))

In [ ]:
from uvera_ml.eval.report import build_summary
summary = build_summary("/tmp/uvera/reports", OUT)
print(json.dumps(summary["available"], indent=1))
print((OUT / "reports" / "summary.md").read_text(encoding="utf-8"))

In [ ]:
# Last step: package the results. Download the zip from the Output tab and send it to Rubab.
budget.check("done")
zip_outputs(OUT)